# Init sweep: which first-layer neuron captures u_0, and why

Analysis notebook for the two sweeps produced by `python -m layer_lenses.init_sweep`
(see that module's docstring and `results/init_sweep/run_all.sh`).

**Part A, invariance.** A few inits, each trained with 3 data samples x 2 batch sizes on the
same ODT, first-layer trajectory stored. Question: is the neuron that captures u_0 a property
of the init alone, for a fixed optimizer?

**Part B, init sweep.** ~200 inits, one run each, full parameter trajectory stored. Question:
what property of the init picks the winner?

Both use constant-LR SGD with momentum and stop when the trailing one-epoch training loss
falls below 0.3. Snapshots are indexed by optimizer step (every 100 steps), so runs with
different batch sizes are compared at equal flow time. "Capture" means |cos(w_k, u_0)| >= 0.95,
either sign. Because two neurons often cross within a few hundred steps of each other, the
analysis works with the **leading set** (everyone crossing within `TIE_WINDOW_STEPS` of the
first) as well as the single first crosser.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from layer_lenses.init_sweep import (
    FIRST_LAYER_KEY,
    SweepRunSpec,
    build_data,
    build_init_model,
    first_layer_agreement,
    init_neuron_features,
    load_run,
    load_tree,
    part_a_invariance_table,
    signed_cosines_with_direction,
    sweep_capture_table,
    alignment_rate_flow_difference,
    alignment_rate_sensitivity,
    winner_prediction_cv,
    winner_rank_table,
)
from layer_lenses.relu_training import checkpoint_model_from_state
import torch

PART_A_DIR = Path("../results/init_sweep/partA")
PART_B_DIR = Path("../results/init_sweep/partB")
REFERENCE_DATA_SEED = 5178
REFERENCE_BATCH_SIZE = 10_000
THRESHOLD = 0.95
TIE_WINDOW_STEPS = 500
DEVICE = "cuda"

## Part A: invariance of the u_0 capture to data sample and batch size

In [ ]:
per_init, per_run = part_a_invariance_table(
    PART_A_DIR,
    reference_data_seed=REFERENCE_DATA_SEED,
    reference_batch_size=REFERENCE_BATCH_SIZE,
    threshold=THRESHOLD,
    tie_window_steps=TIE_WINDOW_STEPS,
)
print(f"{len(per_run)} runs over {len(per_init)} inits")
display(per_init)
print("inits where every run has the same first crosser:      ", int(per_init.all_same_winner.sum()), "/", len(per_init))
print("inits where every leading set equals the reference's:  ", int(per_init.all_leading_sets_equal_ref.sum()), "/", len(per_init))
print("inits where every leading set meets the reference's:   ", int(per_init.all_leading_sets_meet_ref.sum()), "/", len(per_init))

In [ ]:
cols = ["init_seed", "data_seed", "batch_size", "final_step", "final_loss", "winner", "winner_step",
        "runner_up", "gap_steps", "leading_set", "on_direction_end",
        "min_row_cos_vs_ref_at_cross", "min_row_cos_vs_ref_end", "median_row_cos_vs_ref_end"]
display(per_run[cols].sort_values(["init_seed", "data_seed", "batch_size"]).reset_index(drop=True))

Row-wise cosine between each run's first layer and the reference run's, at equal step count.
A value of 1 means the two runs are the same trajectory neuron for neuron; the vertical line marks
the reference winner's crossing step.

In [ ]:
runs_a = {r.run_id: load_run(r.path) for r in per_run.itertuples()}
inits = sorted(per_init.init_seed)
fig, axes = plt.subplots(1, len(inits), figsize=(4.2 * len(inits), 3.6), sharey=True, squeeze=False)
for ax, init_seed in zip(axes[0], inits):
    grp = per_run[per_run.init_seed == init_seed]
    ref = grp[(grp.data_seed == REFERENCE_DATA_SEED) & (grp.batch_size == REFERENCE_BATCH_SIZE)].iloc[0]
    for r in grp.itertuples():
        if r.run_id == ref.run_id:
            continue
        ag = first_layer_agreement(runs_a[ref.run_id], runs_a[r.run_id])
        ax.plot(ag.step, ag.min_cos, label=f"data {r.data_seed}, bs {r.batch_size}", lw=1.4)
    ax.axvline(ref.winner_step, color="k", ls="--", lw=1)
    ax.set_title(f"init {init_seed}: ref winner {ref.winner}")
    ax.set_xlabel("step"); ax.grid(alpha=0.25); ax.set_ylim(-0.05, 1.02)
axes[0, 0].set_ylabel("min over neurons of row-wise cos vs reference")
axes[0, -1].legend(fontsize=7)
fig.tight_layout(); plt.show()

## Part B: the init sweep

One run per init. First the outcome statistics: how often there is a clear winner, how often a
near-tie, and when captures happen.

In [ ]:
table_b = sweep_capture_table(PART_B_DIR, threshold=THRESHOLD, tie_window_steps=TIE_WINDOW_STEPS)
tree_b = load_tree(PART_B_DIR / "tree.npz")
print(f"{len(table_b)} runs; stopped by loss: {(table_b.stopped_by == 'stop_loss').sum()}, by max_steps: {(table_b.stopped_by == 'max_steps').sum()}")
print(f"runs with a u_0 capture: {(table_b.winner >= 0).sum()} / {len(table_b)}")
print("leading-set size distribution:", table_b.n_leading.value_counts().sort_index().to_dict())
print("neurons on u_0 at loss 0.3, distribution:", table_b.n_on_direction_end.value_counts().sort_index().to_dict())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].hist(table_b.winner_step[table_b.winner >= 0], bins=30); axes[0].set_title("first crossing step")
axes[1].hist(table_b.gap_steps[table_b.runner_up >= 0], bins=30); axes[1].set_title("gap to runner-up (steps)")
axes[2].hist(table_b.final_step, bins=30); axes[2].set_title("steps to loss 0.3")
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()
display(table_b[["init_seed", "final_step", "final_loss", "winner", "winner_step", "winner_cos_end",
                 "runner_up", "gap_steps", "leading_set", "n_on_direction_end"]].head(12))

### Per-neuron init features and where the winner ranks

For every init, 40 rows of features at initialization and at the end of the plateau (first
snapshot with trailing loss <= 0.69), plus the outcome. The population gradient and lens at
init are computed on a fresh 100k-point sample from the same ODT, independent of the training
data. This cell is the expensive one (a few seconds per init).

In [ ]:
FEATURE_N = 100_000
FEATURE_DATA_SEED = 999_001
_spec0 = load_run(table_b.path.iloc[0])["spec"]
x_feat, y_feat = build_data(SweepRunSpec(**{**_spec0.__dict__, "n_train": FEATURE_N, "data_seed": FEATURE_DATA_SEED}), tree_b)

frames = []
for i, r in enumerate(table_b.itertuples()):
    run = load_run(r.path)
    frames.append(init_neuron_features(run, tree_b, x_feat, y_feat, device=DEVICE,
                                       threshold=THRESHOLD, tie_window_steps=TIE_WINDOW_STEPS))
    if (i + 1) % 25 == 0:
        print(f"  features for {i + 1}/{len(table_b)} inits")
features = pd.concat(frames, ignore_index=True)
features.to_csv(PART_B_DIR / "init_neuron_features.csv", index=False)
print(features.shape)

In [ ]:
# Where does the first crosser rank among its 39 siblings, feature by feature?
rank_first = winner_rank_table(features, outcome_col="is_winner")
display(rank_first)

In [ ]:
# Same question for membership in the leading set (near-ties count as winners): compare the
# leading-set members' feature distribution against everyone else, pooled over inits.
lead = features[features.in_leading_set == 1]
rest = features[features.in_leading_set == 0]
feat_cols = [c for c in rank_first.feature]
rows = []
for c in feat_cols:
    a, b = lead[c].to_numpy(float), rest[c].to_numpy(float)
    # rank-biserial: P(lead > rest) - P(lead < rest), from the pooled ranks
    pooled = pd.Series(np.concatenate([a, b])).rank().to_numpy()
    ra = pooled[: len(a)]
    auc = (ra.sum() - len(a) * (len(a) + 1) / 2) / (len(a) * len(b))
    rows.append({"feature": c, "n_lead": len(a), "n_rest": len(b), "P(lead > rest)": auc,
                 "lead_median": np.median(a), "rest_median": np.median(b)})
lead_vs_rest = pd.DataFrame(rows).sort_values("P(lead > rest)", key=lambda s: (s - 0.5).abs(), ascending=False)
display(lead_vs_rest)

In [ ]:
# Winner percentile histograms for the strongest features
top = rank_first.head(6).feature.tolist()
fig, axes = plt.subplots(1, len(top), figsize=(3.2 * len(top), 3.0), sharey=True)
for ax, c in zip(np.atleast_1d(axes), top):
    pct = []
    for _, g in features.groupby("init_seed"):
        if g.is_winner.sum() != 1:
            continue
        v = g[c].to_numpy(float); w = v[g.is_winner.to_numpy() == 1][0]
        pct.append(((v < w).sum() + 0.5 * ((v == w).sum() - 1)) / (len(v) - 1))
    ax.hist(pct, bins=10, range=(0, 1)); ax.set_title(c, fontsize=8); ax.grid(alpha=0.25)
axes[0].set_ylabel("# inits")
fig.suptitle("winner's percentile rank within its own init", y=1.02)
fig.tight_layout(); plt.show()

### Can the winner be predicted from the init?

Leave-one-init-out conditional logit: a linear score over per-neuron features, softmax over
the 40 neurons of an init, trained on the other inits. Reported: how often the top-scored neuron
is the first crosser (chance 1/40), how often it is in the leading set, and the median rank of
the true winner. Signed features enter as absolute values. Three feature sets: static features
at initialization only; features at the end of the plateau, which are a deterministic function
of the init under the flow; and both.

In [ ]:
features["abs_neg_grad_cos_u0_init"] = features["neg_grad_cos_u0_init"].abs()
INIT_COLS = ["abs_cos_u0_init", "norm_init", "fanout_norm_init", "grad_norm_init", "grad_abs_cos_common_init",
             "grad_level4_fraction_init", "abs_neg_grad_cos_u0_init", "lens_abs_mean_init", "lens_std_init"]
PLATEAU_COLS = ["abs_cos_u0_plateau", "norm_ratio_plateau", "disp_norm_plateau", "abs_disp_cos_u0_plateau", "fanout_norm_plateau"]
pred_table, pred_weights = winner_prediction_cv(features, {
    "abs_cos_u0_init only": ["abs_cos_u0_init"],
    "init features": INIT_COLS,
    "plateau features": PLATEAU_COLS,
    "init + plateau": INIT_COLS + PLATEAU_COLS,
})
display(pred_table)
print("weights, init-only model (features z-scored within init):")
for k, v in sorted(pred_weights["init features"].items(), key=lambda kv: -abs(kv[1])):
    print(f"  {k:28s} {v:+.3f}")

### Second-order sensitivity of the alignment rate at init

For neuron k, `A0` is the time derivative of |cos(w_k, u_0)| under population gradient flow
and `B0` the time derivative of log ||w_k||, both evaluated at initialization on the fresh
feature sample. `A_dot` and `B_dot` are their derivatives along the flow, i.e. the second time
derivatives of alignment and log-norm at init, computed exactly by forward-mode differentiation
over the reverse-mode gradient (`alignment_rate_sensitivity`). Two halves of the sample are
averaged, and their agreement is reported. A finite difference along the flow with a step of
1% of ||theta|| (`alignment_rate_flow_difference`) crosses ReLU gate boundaries and so includes
the gate-flip part of the population Hessian that the piecewise-linear derivative cannot see;
its agreement with the derivative says how much that part matters here.

In [ ]:
half = len(x_feat) // 2
template = build_init_model(load_run(table_b.path.iloc[0])["spec"])
rows = []
for i, r in enumerate(table_b.itertuples()):
    run = load_run(r.path)
    sd = {n: torch.from_numpy(a[0]).float() for n, a in run["weights"].items()}
    m0 = checkpoint_model_from_state(template, sd)
    ha = alignment_rate_sensitivity(m0, x_feat[:half], y_feat[:half], tree_b.w_list[0], device=DEVICE)
    hb = alignment_rate_sensitivity(m0, x_feat[half:], y_feat[half:], tree_b.w_list[0], device=DEVICE)
    fd = alignment_rate_flow_difference(m0, x_feat, y_feat, tree_b.w_list[0], rel_step=1e-2, device=DEVICE)
    for k in range(len(ha)):
        rows.append({"init_seed": r.init_seed, "neuron": k,
                     "A0": (ha.A0[k] + hb.A0[k]) / 2, "B0": (ha.B0[k] + hb.B0[k]) / 2,
                     "A_dot": (ha.A_dot[k] + hb.A_dot[k]) / 2, "B_dot": (ha.B_dot[k] + hb.B_dot[k]) / 2,
                     "A_dot_half_a": ha.A_dot[k], "A_dot_half_b": hb.A_dot[k],
                     "A_dot_fd": fd.A_dot_fd[k], "B_dot_fd": fd.B_dot_fd[k]})
sens = pd.DataFrame(rows)
sens.to_csv(PART_B_DIR / "alignment_sensitivity.csv", index=False)
M = features.merge(sens, on=["init_seed", "neuron"]); M["abs_A0"] = M.A0.abs()
spearman = lambda a, b: a.rank().corr(b.rank())
print("median within-init Spearman across the 40 neurons:")
print(f"  A_dot, two 50k halves:                  {np.median([spearman(g.A_dot_half_a, g.A_dot_half_b) for _, g in M.groupby('init_seed')]):.3f}")
print(f"  A_dot, derivative vs 1% finite step:    {np.median([spearman(g.A_dot, g.A_dot_fd) for _, g in M.groupby('init_seed')]):.3f}")
print(f"  B_dot, derivative vs 1% finite step:    {np.median([spearman(g.B_dot, g.B_dot_fd) for _, g in M.groupby('init_seed')]):.3f}")
rank_dyn = winner_rank_table(M)
display(rank_dyn[rank_dyn.feature.isin(["A0", "abs_A0", "A_dot", "B0", "B_dot", "A_dot_fd", "B_dot_fd", "abs_cos_u0_init", "grad_norm_init", "disp_norm_plateau"])])
DYN_COLS = ["abs_A0", "B0", "A_dot", "B_dot"]
pred_dyn, _ = winner_prediction_cv(M, {
    "A_dot alone": ["A_dot"], "B_dot alone": ["B_dot"], "dynamical (4)": DYN_COLS,
    "static init (9)": INIT_COLS, "static + dynamical": INIT_COLS + DYN_COLS,
    "plateau (5)": PLATEAU_COLS, "static + dynamical + plateau": INIT_COLS + DYN_COLS + PLATEAU_COLS,
})
display(pred_dyn)

### When is the winner decided?

The same rates evaluated at later snapshots of each run. For every quantity and snapshot, the
raw hit rate of "the neuron with the largest value is the winner" (and "is in the leading
set"), pooled over inits. Snapshot steps are absolute; the median winner crosses near step
10000, so step 5000 is about halfway to capture. No fitting here.

In [ ]:
STEPS = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]
runs_b = {r.init_seed: load_run(r.path) for r in table_b.itertuples()}
outcome = features[["init_seed", "neuron", "is_winner", "in_leading_set"]]
rows = []
for step in STEPS:
    for init, run in runs_b.items():
        idx_s = int(np.nonzero(run["snapshot_steps"] == step)[0][0])
        sd = {n: torch.from_numpy(a[idx_s]).float() for n, a in run["weights"].items()}
        out = alignment_rate_sensitivity(checkpoint_model_from_state(template, sd), x_feat, y_feat, tree_b.w_list[0], device=DEVICE)
        W = sd[FIRST_LAYER_KEY].numpy()
        cos = np.abs(W @ tree_b.w_list[0]) / np.linalg.norm(W, axis=1) / np.linalg.norm(tree_b.w_list[0])
        disp = np.linalg.norm(W - run["weights"][FIRST_LAYER_KEY][0], axis=1)
        for k in range(len(out)):
            rows.append({"step": step, "init_seed": init, "neuron": k, "A0": out.A0[k], "A_dot": out.A_dot[k],
                         "B0": out.B0[k], "B_dot": out.B_dot[k], "abs_cos_u0": cos[k], "disp": disp[k]})
over_time = pd.DataFrame(rows).merge(outcome, on=["init_seed", "neuron"])
over_time.to_csv(PART_B_DIR / "rates_over_time.csv", index=False)
quantities = ["A0", "A_dot", "B0", "B_dot", "abs_cos_u0", "disp"]
hit = []
for step in STEPS:
    d = over_time[over_time.step == step]
    row = {"step": step}
    for q in quantities:
        top1 = lead = n = 0
        for _, g in d.groupby("init_seed"):
            if g.is_winner.sum() != 1:
                continue
            best = g.loc[g[q].idxmax()]; n += 1
            top1 += int(best.is_winner == 1); lead += int(best.in_leading_set == 1)
        row[f"{q} top1"] = round(top1 / n, 2); row[f"{q} lead"] = round(lead / n, 2)
    hit.append(row)
display(pd.DataFrame(hit).set_index("step"))
fig, ax = plt.subplots(figsize=(7, 3.6))
H = pd.DataFrame(hit).set_index("step")
for q in quantities:
    ax.plot(H.index, H[f"{q} top1"], marker="o", label=q)
ax.axhline(1 / 40, color="k", ls=":", lw=1, label="chance")
ax.set_xlabel("snapshot step"); ax.set_ylabel("P(argmax is the winner)"); ax.grid(alpha=0.25); ax.legend(fontsize=8)
ax.set_title("how early does each quantity identify the winner?")
fig.tight_layout(); plt.show()

### Trajectories for a few inits

Signed cosine with u_0 of every first-layer neuron over steps, winner highlighted.

In [ ]:
show = table_b[table_b.winner >= 0].head(4)
fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.4), sharey=True, squeeze=False)
for ax, r in zip(axes[0], show.itertuples()):
    run = load_run(r.path)
    cos = signed_cosines_with_direction(run["weights"][FIRST_LAYER_KEY], tree_b.w_list[0])
    lead = [int(k) for k in r.leading_set.split(",") if r.leading_set]
    for k in range(cos.shape[1]):
        ax.plot(run["snapshot_steps"], cos[:, k], color="tab:red" if k in lead else "0.75",
                lw=1.6 if k in lead else 0.6, zorder=2 if k in lead else 1)
    ax.axhline(THRESHOLD, color="k", ls="--", lw=0.8); ax.axhline(-THRESHOLD, color="k", ls="--", lw=0.8)
    ax.set_title(f"init {r.init_seed}: leading set {lead}"); ax.set_xlabel("step"); ax.grid(alpha=0.25)
axes[0, 0].set_ylabel("cos(w_k, u_0)")
fig.tight_layout(); plt.show()